# Hướng dẫn cài đặt và sử dụng Fiftyone

### Trước tiên phải có: `Docker Desktop`, `Python 3.11.0`

Cài đặt module cần thiết, tạo image qdrant mới trên docker `(lưu ý phải mở Docker Desktop trước)`, import module

In [7]:
! pip install fiftyone
! pip install torch torchvision torchaudio
! pip install qdrant-client

In [8]:
! docker pull qdrant/qdrant
! docker run -p 6333:6333 qdrant/qdrant

zsh:1: command not found: docker
zsh:1: command not found: docker


In [21]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.utils.image as fui
import numpy as np
from glob import glob
import json
import os
import csv

Tạo dataset mới (dòng đầu): nhập link dẫn đến folder chứa các tên Keyframes, thay đổi tên theo ý muốn

Tải lại dataset đã tạo (dòng sau): nhập tên dataset muốn load lại `(khi đã có dataset để dự thi thì chỉ chạy dòng sau để load dataset)`

In [10]:
dataset = fo.Dataset.from_images_dir('/Volumes/owo/AIC2024/kfs', name="test", tags=None, recursive=True, persistent=True)

ValueError: Dataset name 'test' is not available

In [22]:
dataset = fo.load_dataset("test") # Main dataset's name is "AIC-2024"

Dòng phụ: Xóa dataset nếu cần

In [ ]:
print(fo.list_datasets())

['AIC2024', 'AICDisplay']


In [ ]:
fo.delete_dataset("test")

Bật/Đóng Fiftyone Web

In [39]:
session = fo.launch_app(dataset, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


In [40]:
session.show()

In [24]:
session.close()

### Trích xuất thêm thông tin tên của video và frameid
Thông tin `video` và `frameid` sẽ được lấy từ tên của tập tin keyframe.

In [14]:
for sample in dataset:
    _, sample['video'], sample['frameid'] = sample['filepath'][:-4].rsplit('/', 2)
    sample.save()

### Thêm thông tin từ CSV map keyframe ###
Thông tin `frame_order` và `time_order` sẽ được lấy từ CSV map keyframe.

In [17]:
filepaths = glob("/Volumes/owo/AIC2024/mapkfs/*.csv")
all_frame_detail = {}
#The dictionary stores dictionaries of videos in which they store details of keyframes


for filepath in filepaths:
    all_frame_detail[filepath[-12:-4]] = {}
    with open(filepath, "r", newline='') as map_keyframe:
        reader = csv.reader(map_keyframe)
        next(reader)
        for row in reader: 
            all_frame_detail[filepath[-12:-4]][row[0]] = [row[1], row[2], row[3]] #time_order, fps, frame_order respectively

for sample in dataset:
    key = str(int(sample["frameid"])) #Omit insignificant zeroes
    sample["time_order"] = str(all_frame_detail[sample["video"]][key][0])
    sample["frame_order"] = str(all_frame_detail[sample["video"]][key][2])
    sample.save()

### Thêm link video và ngày phát hành của video ###
Thông tin `youtube_link` và `publish_date` sẽ được lấy từ file JSON metadata của video.

In [18]:
for sample in dataset:
    json_path = f"/Volumes/owo/AIC2024/metadata/{sample['video']}.json"
    with open(json_path, "r", encoding="utf-8") as jsonfile:
        metadata = json.load(jsonfile)
    sample["publish_date"] = metadata["publish_date"]
    sample["youtube_link"] = metadata["watch_url"] + f"&t={sample['time_order']}s" #Added time suffix to jump to keyframe's time
    sample.save()

### Thêm thời gian của keyframe tính theo ms
Thông tin `time_in_ms` được tính dựa trên `time_order`

In [20]:
for sample in dataset:
    sample["time_in_ms"] = str(float(sample["time_order"]) * 1000)
    sample.save()

Bạn có thể xem `Sample` đầu tiên của `Dataset` bằng lệnh sau:

In [23]:
print(dataset.first())

<Sample: {
    'id': '68a73360c088828c0e51ca1b',
    'media_type': 'image',
    'filepath': '/Volumes/owo/AIC2024/kfs/Keyframes_L01/keyframes/L01_V001/001.jpg',
    'tags': [],
    'metadata': None,
    'video': 'L01_V001',
    'frameid': '001',
    'time_order': '0.0',
    'frame_order': '0',
    'publish_date': '31/10/2023',
    'youtube_link': 'https://youtube.com/watch?v=1yHly8dYhIQ&t=0.0s',
    'time_in_ms': '0.0',
}>


### Thêm thông tin CLIP embedding.

In [32]:
all_keyframe = glob('/Volumes/owo/AIC2024/kfs/*/keyframes/*/*.jpg')
video_keyframe_dict = {}
all_video = glob('/Volumes/owo/AIC2024/kfs/*/keyframes/*')  
all_video = [v.rsplit('/',1)[-1] for v in all_video]

Tạo dictionary `video_keyframe_dict` với `video_keyframe_dict[video]` thông tin danh sách `keyframe` của `video` 

In [33]:
for k in all_keyframe:
    _, vid, kf = k[:-4].rsplit('/',2)
    if vid not in video_keyframe_dict.keys():
        video_keyframe_dict[vid] = [kf]
    else:
        video_keyframe_dict[vid].append(kf)

Do thông tin vector CLIP embedding được cung cấp được lưu theo từng video nhầm mục đích tối ưu thời gian đọc dữ liệu. Cần sort lại danh sách `keyframe` của từng `video` để đảm bảo thứ tự đọc đúng với vector embedding được cung cấp.

In [34]:
for k,v in video_keyframe_dict.items():
    video_keyframe_dict[k] = sorted(v)

Tạo dictionary `embedding_dict` với `embedding_dict[video][keyframe]` lưu thông tin vector CLIP embedding của `keyframe` trong `video` tương ứng

In [35]:
embedding_dict = {}
for v in all_video:
    clip_path = f'/Volumes/owo/AIC2024/clip32/{v}.npy'
    a = np.load(clip_path)
    embedding_dict[v] = {}
    for i,k in enumerate(video_keyframe_dict[v]):
        embedding_dict[v][k] = a[i]


Tạo danh sách `clip_embedding` ứng với danh sách `sample` trong `dataset`.

In [36]:
clip_embeddings = []
for sample in dataset:
    clip_embedding = embedding_dict[sample['video']][sample['frameid']]
    clip_embeddings.append(clip_embedding)


Lưu/tải lên `clip_embeddings`

In [37]:
# Lưu
np.save("data.npy", np.array(clip_embeddings))

In [3]:
# Tải lên
clip_embeddings = np.load("clip-vit-l14-embeddings.npy")

Đây là phần tính toán key để truy vấn

In [38]:
qdrant_idx = fob.compute_similarity(
    dataset,
    model="clip-vit-base32-torch",
    embeddings=clip_embeddings,          # precomputed image embeddings
    brain_key="img_sim",
    progress=True,
)

In [16]:
dataset.delete_brain_run("img_sim_qdrant")

# Những phần dưới đây là tùy chọn (lưu ý không chạy giảm kích cỡ hình ảnh)

### Thêm thông tin kết quả của object detection

In [6]:
for sample in dataset:
    object_path = f"D:\\AIC2024\\objects\\{sample['video']}\\{sample['frameid']}.json"
    with open(object_path) as jsonfile:
        det_data = json.load(jsonfile)
    detections = []
    for cls, box, score in zip(det_data['detection_class_entities'], det_data['detection_boxes'], det_data['detection_scores']):
        scoref = float(score)
        # Only add objects with confidence >= 0.6
        if scoref < 0.4: continue
        # Convert to [top-left-x, top-left-y, width, height]
        boxf = [float(box[1]), float(box[0]), float(box[3]) - float(box[1]), float(box[2]) - float(box[0])]
        
        detections.append(
            fo.Detection(
                label=cls,
                bounding_box= boxf,
                confidence=float(score)
            )
        )
    sample["object_faster_rcnn"] = fo.Detections(detections=detections)
    sample.save()

# Calc scoref first then append box later; push score restriction up to 0.6

### Giảm chất lượng hình ảnh từ 1280x700 xuống còn 640x360 (giảm 2 lần độ lớn dataset) `KHÔNG ĐƯỢC CHẠY KHI CHƯA CÓ THÔNG BÁO`

In [6]:
fui.transform_images(
    dataset,
    size=(640, 360),
    delete_originals=True,
    progress=True,
)

 100% |███████████| 352558/352558 [1.7h elapsed, 0s remaining, 61.5 samples/s]      


In [8]:
for sample in dataset:
    sample["time_in_ms"] =str(int(float(sample["time_order"]) * 1000))
    sample["check_code"] = f"{sample['video']}-{sample['time_in_ms']}"
    sample.save()

In [18]:
view = dataset.sort_by_similarity("jellyfish", k=100, brain_key="img_sim_qdrant")
session.view = view

UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Wrong input: Vector dimension error: expected dim: 768, got 512"},"time":0.002955695}'